In [1]:
!pip install pandas h5py xarray netCDF4 eumdac satpy scikit-learn netcdf4 matplotlib cartopy
!pip install -U "pycomlink==0.6.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 32.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 115.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 150.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 18.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.6/679.6 kB 46.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 118.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [satpy]m19/20 [satpy]y]le]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 9.7 MB/s  0:00:026m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 144.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 90.9 MB/s  0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.15.1
    Uninstalling h5py-3.15.1:
      Successfully uninstalled h5py-3.15.1
   

In [1]:
!python inspect_essd_three_netcdf.py

Inspecting ESSD publication NetCDF files
No files will be modified.

ESSD_publication_dataset/CZ_CML_PREPROCESSED_1MIN_20200101_20211231.nc
Size: 3,516.91 MiB

DIMENSIONS
----------------------------------------------------------------------------------------------------
time                                1,052,640
sublink_id                          2
cml_id                              2,219

COORDINATES
----------------------------------------------------------------------------------------------------

detection_limit_mmhr
    dims : ('cml_id',)
    shape: (2219,)
    dtype: float64
    attrs:
        description: 'Detection limit based on ITU-R 2003'
        units: 'mm/hr'

time
    dims : ('time',)
    shape: (1052640,)
    dtype: datetime64[ns]
    attrs:
        (none)

sublink_id
    dims : ('sublink_id',)
    shape: (2,)
    dtype: int64
    attrs:
        (none)

length
    dims : ('cml_id',)
    shape: (2219,)
    dtype: float32
    attrs:
        (none)

site_a_latitude
 

In [5]:
!python harmonize_essd_cml_radar_metadata_v2.py \
    --root ESSD_publication_dataset 

ESSD metadata: 100%|████████████████| 10/10 [01:01<00:00,  6.16s/step, complete]

Metadata harmonization complete.
Preprocessed CML : /home/jovyan/ESSD_publication_dataset/CZ_CML_PREPROCESSED_1MIN_20200101_20211231.nc
CML rain rate    : /home/jovyan/ESSD_publication_dataset/CZ_CML_RAINRATE_5MIN_20200101_20211231.nc
Radar CML paths  : /home/jovyan/ESSD_publication_dataset/CZ_RADAR_CMLPATH_5MIN_20200101_20211231.nc
Scientific array values and dimensions were not recomputed or changed.
Validated temporary copies were atomically installed over the originals.


In [7]:
import xarray as xr

ds= xr.open_dataset("ESSD_publication_dataset/CZ_CML_RAINRATE_5MIN_20200101_20211231.nc")
ds.kr_model

<xarray.DataArray 'kr_model' (kr_model: 2)> Size: 64B
array(['ITU_2003', 'ITU_2005'], dtype='<U8')
Coordinates:
  * kr_model  (kr_model) <U8 64B 'ITU_2003' 'ITU_2005'
Attributes:
    long_name:  Attenuation-rain-rate relation model
    comment:    Identifier of the ITU attenuation-rain-rate coefficient set u...

In [ ]:
!cd /home/jovyan

!python -u /home/jovyan/prepare_essd_publication_datasets_fast.py \
  --cml-preprocessed /home/jovyan/cml_preprocessed_v3.nc \
  --cml-raw /home/jovyan/CML_S2020E2021_1MIN_CZ.nc \
  --radar /home/jovyan/RADAR_S2021E2021_5MIN_CZ.nc \
  --gauge /home/jovyan/CHMI_10MIN_v1.nc \
  --cml-rainrate /home/jovyan/cml_rainrate_S2020_E2021_valid_cml_radar.nc \
  --out-dir /home/jovyan/ESSD_publication_dataset \
  --time-chunk-1min 10080 \
  --time-chunk-5min 2016 \
  --time-chunk-10min 1008 \
  --cml-chunk 64 \
  --compression-level 1 \
  --io-workers 4 \
  --overwrite

ESSD PUBLICATION DATASET PREPARATION
Master CML file    : /home/jovyan/cml_preprocessed_v3.nc
Master lower limit : 2020-01-01 00:00:00
Master upper limit : 2021-12-31 23:59:00
Period tag         : 20200101_20211231
/home/jovyan/prepare_essd_publication_datasets_fast.py:71: UserWarning: The specified chunks separate the stored chunks along dimension "cml_id" starting at index 64. This could degrade performance. Instead, consider rechunking after loading.
  return xr.open_dataset(path, chunks=chunks)
/home/jovyan/prepare_essd_publication_datasets_fast.py:71: UserWarning: The specified chunks separate the stored chunks along dimension "time_5min" starting at index 2016. This could degrade performance. Instead, consider rechunking after loading.
  return xr.open_dataset(path, chunks=chunks)

Scanning R_mean_5min for CMLs with at least one finite rain-rate value ...
Rain-rate CML IDs in source : 1,133
CML IDs with valid rain rate: 1,132

Writing: /home/jovyan/ESSD_publication_dataset/CZ_CML

In [1]:
from pathlib import Path

import numpy as np
import xarray as xr


FILES = {
    "Raw CML dataset": Path(
        "/home/jovyan/ESSD_publication_dataset/"
        "CZ_CML_RAW_1MIN_20200101_20211231.nc"
    ),
    "Quality controlled CML dataset": Path(
        "/home/jovyan/ESSD_publication_dataset/"
        "CZ_CML_PREPROCESSED_1MIN_20200101_20211231.nc"
    ),
}

E_BAND_THRESHOLD_GHZ = 70.0


def calculate_e_band_fraction(path: Path) -> dict:
    with xr.open_dataset(path) as ds:
        if "frequency" not in ds:
            raise KeyError(f"{path}: variable 'frequency' was not found.")

        frequency = ds["frequency"].astype("float64")

        # Convert MHz to GHz when necessary.
        finite_values = frequency.values[np.isfinite(frequency.values)]
        if finite_values.size == 0:
            raise ValueError(f"{path}: no finite frequency values.")

        if np.nanmedian(finite_values) > 1000:
            frequency_ghz = frequency / 1000.0
        else:
            frequency_ghz = frequency

        if "sublink_id" in frequency_ghz.dims:
            # A physical CML is classified as E band when at least one
            # directional sublink operates above 70 GHz.
            finite_per_cml = frequency_ghz.notnull().any(dim="sublink_id")
            e_band_per_cml = (
                frequency_ghz > E_BAND_THRESHOLD_GHZ
            ).any(dim="sublink_id")
        else:
            finite_per_cml = frequency_ghz.notnull()
            e_band_per_cml = frequency_ghz > E_BAND_THRESHOLD_GHZ

        total_cmls = int(ds.sizes["cml_id"])
        cmls_with_frequency = int(finite_per_cml.sum().item())
        e_band_cmls = int((e_band_per_cml & finite_per_cml).sum().item())

        percentage_all = 100.0 * e_band_cmls / total_cmls
        percentage_valid = (
            100.0 * e_band_cmls / cmls_with_frequency
            if cmls_with_frequency
            else np.nan
        )

        return {
            "total_cmls": total_cmls,
            "cmls_with_frequency": cmls_with_frequency,
            "e_band_cmls": e_band_cmls,
            "percentage_all": percentage_all,
            "percentage_valid": percentage_valid,
        }


for label, path in FILES.items():
    if not path.exists():
        print(f"\n{label}: file not found\n  {path}")
        continue

    result = calculate_e_band_fraction(path)

    print(f"\n{label}")
    print("=" * len(label))
    print(f"Total CMLs                    : {result['total_cmls']:,}")
    print(f"CMLs with frequency metadata : {result['cmls_with_frequency']:,}")
    print(f"E band CMLs above 70 GHz     : {result['e_band_cmls']:,}")
    print(f"E band percentage, all CMLs  : {result['percentage_all']:.2f}%")
    print(
        "E band percentage, CMLs with frequency metadata: "
        f"{result['percentage_valid']:.2f}%"
    )


Raw CML dataset
Total CMLs                    : 2,580
CMLs with frequency metadata : 2,572
E band CMLs above 70 GHz     : 904
E band percentage, all CMLs  : 35.04%
E band percentage, CMLs with frequency metadata: 35.15%

Quality controlled CML dataset
Total CMLs                    : 2,219
CMLs with frequency metadata : 2,219
E band CMLs above 70 GHz     : 902
E band percentage, all CMLs  : 40.65%
E band percentage, CMLs with frequency metadata: 40.65%


In [16]:
!cd /home/jovyan

!python -u /home/jovyan/cml_qc_filter_standalone_v4_3.py \
  --qc-nc /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_GRAF_RSTD_WETDRY.nc \
  --out-nc /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_STRICT_MINIMAL.nc \
  --summary-json /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_STRICT_MINIMAL_summary.json \
  --summary-csv /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_STRICT_MINIMAL_filtering.csv \
  --link-pass-var qc_final_link_pass \
  --sublink-pass-var qc_final_sublink_pass \
  --valid-var qc_final_valid \
  --tx-var tx \
  --rx-var rx \
  --trsl-var trsl_qc \
  --frequency-var frequency \
  --polarization-var polarization \
  --length-var length \
  --time-dim time \
  --sublink-dim sublink_id \
  --cml-dim cml_id \
  --min-valid-fraction 0 \
  --min-valid-observations 1 \
  --time-chunk 10080 \
  --link-batch 16 \
  --link-chunk 16 \
  --compression-level 1 \
  --overwrite

Running cml_qc_filter_standalone_v4_3.py version 4.3
Scanning sample-level validity ...
  validity 10,080/1,052,640
  validity 20,160/1,052,640
  validity 30,240/1,052,640
  validity 40,320/1,052,640
  validity 50,400/1,052,640
  validity 60,480/1,052,640
  validity 70,560/1,052,640
  validity 80,640/1,052,640
  validity 90,720/1,052,640
  validity 100,800/1,052,640
  validity 110,880/1,052,640
  validity 120,960/1,052,640
  validity 131,040/1,052,640
  validity 141,120/1,052,640
  validity 151,200/1,052,640
  validity 161,280/1,052,640
  validity 171,360/1,052,640
  validity 181,440/1,052,640
  validity 191,520/1,052,640
  validity 201,600/1,052,640
  validity 211,680/1,052,640
  validity 221,760/1,052,640
  validity 231,840/1,052,640
  validity 241,920/1,052,640
  validity 252,000/1,052,640
  validity 262,080/1,052,640
  validity 272,160/1,052,640
  validity 282,240/1,052,640
  validity 292,320/1,052,640
  validity 302,400/1,052,640
  validity 312,480/1,052,640
  validity 322,560/1,0

In [22]:
import xarray as xr
import pandas as pd 
ds = pd.read_csv("CML META/metadata_paired_2021.csv",sep=";")["pol"].unique()
ds

<StringArray>
['Vertical']
Length: 1, dtype: str

In [28]:
!cd /home/jovyan

!python -u /home/jovyan/cml_estimate_and_grid_cml_radar_gauge_merged_parallel_timewindow_v2.py \
  --cml-nc /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_STRICT_MINIMAL.nc \
  --trsl-var trsl_qc \
  --coords-nc /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_GRAF_RSTD_WETDRY.nc \
  --radar-nc /home/jovyan/RADAR_S2021E2021_5MIN_CZ.nc \
  --gauge-nc /home/jovyan/CHMI_10MIN_v1.nc \
  --estimation-nc /home/jovyan/cml_rainrate_baseline_kr_waa_methods_june_v2.nc \
  --gridded-nc /home/jovyan/cml_rainfall_gridded_june_v2.nc \
  --log-csv /home/jovyan/cml_rainrate_baseline_kr_waa_methods_june_v2_log.csv \
  --tstart "2020-01-01 00:00:00" \
  --tend "2021-12-31 23:59:00" \
  --grid-baseline-method pycml_constant \
  --grid-kr-model ITU_2003 \
  --grid-waa-method pastorek_2021 \
  --grid-time-chunk 288 \
  --gauge-time-chunk 144 \
  --batch 32 \
  --cell-chunk 20000 \
  --grid-workers 2 \
  --idw-query-workers 1 \
  --overwrite-estimation \
  --overwrite-grid

Matplotlib is building the font cache; this may take a moment.
Running cml_estimate_and_grid_cml_radar_gauge_merged_parallel_timewindow_v2.py
Filtered-input compatibility only; scientific workflow unchanged

[stage 1/2] Running original rainfall-estimation pipeline ...
Loading datasets lazily...
  Total CMLs             : 2064
  Radar CML ids          : 2580
  Selected CMLs          : 1991
  Sublinks per link      : 2
  TRSL input             : trsl_qc
  Processing window      : 2020-01-01 00:00:00 -> 2021-12-31 23:59:00 (1052640 min)
  Output window          : 2020-01-01 00:00:00 -> 2021-12-31 23:59:00 (1052640 min)
  Radar convention       : exact event script, radar_mmph = radar_raw * 12
  Baseline methods       : pycml_constant, last_dry
  k-R models             : ITU_2003, ITU_2005
  WAA methods            : none, leijnse_2008, pastorek_2021, schleiss_2013
  Write 1-min output     : False
CML links: 100%|██████████████████████████| 1991/1991 [1:05:13<00:00,  1.97s/it]

Processing 

In [17]:
!cd /home/jovyan

!python -u /home/jovyan/cml_compare_single_combo_wet_radar_only.py \
  --rain-nc ESSD_publication_dataset/CZ_CML_RAINRATE_5MIN_20200101_20211231.nc \
  --out-dir ESSD_publication_dataset/CZ_CML_RAINRATE_5MIN_20200101_20211231 \
  --cml-var R_sublink_5min \
  --reference gauge \
  --radar-var R_radar \
  --baseline-method pycml_constant \
  --kr-model ITU_2003 \
  --waa-method pastorek_2021 \
  --wet-only \
  --radar-wet-thr 0.1 \
  --gridsize 50 \
  --max-rain 150


Loading dataset
  file: ESSD_publication_dataset/CZ_CML_RAINRATE_5MIN_20200101_20211231.nc

Selecting CML estimate
  Selected baseline method: pycml_constant on dim 'baseline_method'
  Selected k-R model      : ITU_2003 on dim 'kr_model'
  Selected WAA method     : pastorek_2021 on dim 'waa_method'
  Gauge file        : CHMI_10MIN_v1.nc
  Gauge var         : rr  (10-min amounts -> mm h-1)
  Max gauge dist    : 5 km
Traceback (most recent call last):
  File "/home/jovyan/cml_compare_single_combo_wet_radar_only.py", line 1246, in <module>
    main()
    ~~~~^^
  File "/home/jovyan/cml_compare_single_combo_wet_radar_only.py", line 1156, in main
    ref_da, gauge_mapping = build_gauge_reference(ds, args, dims)
                            ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/home/jovyan/cml_compare_single_combo_wet_radar_only.py", line 496, in build_gauge_reference
    raise ValueError(
    ...<2 lines>...
    )
ValueError: CML site coordinate variable 'site_a_latitude' is not in

In [6]:
import xarray as xr
ds = xr.open_dataset("/home/jovyan/cml_rainrate_S2020_E2021_valid_cml_radar.nc")
ds



<xarray.Dataset> Size: 12GB
Dimensions:          (cml_id: 1133, sublink_id: 2, baseline_method: 1,
                      kr_model: 2, waa_method: 4, time_5min: 105120)
Coordinates:
  * cml_id           (cml_id) <U17 77kB '10001_10385_3108' ... '89221_347241_...
  * sublink_id       (sublink_id) int32 8B 0 1
  * baseline_method  (baseline_method) <U14 56B 'pycml_constant'
  * kr_model         (kr_model) <U8 64B 'ITU_2003' 'ITU_2005'
  * waa_method       (waa_method) <U13 208B 'none' ... 'schleiss_2013'
  * time_5min        (time_5min) datetime64[ns] 841kB 2021-01-01 ... 2021-12-...
Data variables:
    R_sublink_5min   (cml_id, sublink_id, baseline_method, kr_model, waa_method, time_5min) float32 8GB ...
    R_mean_5min      (cml_id, baseline_method, kr_model, waa_method, time_5min) float32 4GB ...
    R_radar          (cml_id, time_5min) float32 476MB ...
    waa_mean         (cml_id, sublink_id, baseline_method, waa_method) float32 36kB ...
    A_obs_mean       (cml_id, sublink_id, baseline_method) float32 9kB ...
    A_rain_mean      (cml_id, sublink_id, baseline_method, waa_method) float32 36kB ...
    length           (cml_id) float32 5kB ...
    frequency        (cml_id, sublink_id) float32 9kB ...
    polarization     (cml_id, sublink_id) <U1 9kB ...
Attributes: (12/23)
    title:                                                 CML rainfall estim...
    processing:                                            per-sublink TRSL +...
    baseline_methods:                                      pycml_constant,las...
    kr_models:                                             ITU_2003,ITU_2005
    waa_methods:                                           none,leijnse_2008,...
    custom_last_dry_baseline:                              closest previous f...
    ...                                                    ...
    valid_cml_radar_filter_cml_kept_csv:                   cml_preprocessed_v...
    valid_cml_radar_filter_radar_kept_csv:                 radar_kept_links_v...
    valid_cml_radar_filter_require_rainfile_radar_signal:  1
    valid_cml_radar_filter_kept_links:                     1133
    valid_cml_radar_filter_dropped_links:                  806
    valid_cml_radar_filter_report_csv:                     cml_rainrate_S2020...

In [8]:
import xarray as xr
ds = xr.open_dataset("/home/jovyan/RADAR_S2021E2021_5MIN_CZ.nc")
ds




<xarray.Dataset> Size: 2GB
Dimensions:  (time: 105120, cml_id: 2580)
Coordinates:
  * time     (time) datetime64[ns] 841kB 2021-01-01 ... 2021-12-31T23:55:00
  * cml_id   (cml_id) <U19 196kB '10001_10066_3107' ... '89221_347241_5030'
Data variables:
    rr       (time, cml_id) float64 2GB ...

In [2]:
!cd /home/jovyan

!python -u /home/jovyan/compare_two_cml_rain_products_same_samples_fast.py \
  --new-rain-nc /home/jovyan/cml_rainrate_baseline_kr_waa_methods_june_v2.nc \
  --old-rain-nc /home/jovyan/cml_rainrate_S2020_E2021_valid_cml_radar.nc \
  --out-dir /home/jovyan/fair_old_vs_strict_qc_v2_fast \
  --cml-var R_mean_5min \
  --radar-var R_radar \
  --radar-source new \
  --baseline-method pycml_constant \
  --kr-model ITU_2003 \
  --waa-method pastorek_2021 \
  --new-label "Strict QC V2" \
  --old-label "Original product" \
  --wet-only \
  --radar-wet-thr 0.1 \
  --min-rain 0 \
  --max-rain 100 \
  --link-chunk 32 \
  --time-chunk 2016 \
  --max-plot-points 500000 \
  --plot-type hexbin \
  --gridsize 50

Opening datasets lazily...
Common links      : 1,056
Common timestamps : 105,120
Time range        : 2021-01-01T00:00:00.000000000 -> 2021-12-31T23:55:00.000000000
Link chunk        : 32
Time chunk        : 2016
Wet-only          : True
Radar wet mask    : R_radar > 0.1 mm h-1
Fair comparison blocks: 100%|███████████████| 1749/1749 [02:48<00:00, 10.40it/s]

FAIR COMMON SAMPLE RESULTS
Exact common N: 2,672,344

NEW PRODUCT
N             2.672344e+06
MBE          -5.399055e-01
MAE           1.185340e+00
RMSE          2.607534e+00
PCC           5.688159e-01
mean_cml      1.008074e+00
mean_radar    1.547979e+00

OLD PRODUCT
N             2.672344e+06
MBE          -5.425431e-01
MAE           1.177872e+00
RMSE          2.568776e+00
PCC           5.612989e-01
mean_cml      1.005436e+00
mean_radar    1.547979e+00

Saved:
/home/jovyan/fair_old_vs_strict_qc_v2_fast/scatter_new_vs_radar.png
/home/jovyan/fair_old_vs_strict_qc_v2_fast/scatter_old_vs_radar.png
/home/jovyan/fair_old_vs_strict_qc_v2_f

In [3]:
!cd /home/jovyan

!python -u /home/jovyan/run_filter_strategy_rainfall_experiment.py \
  --legacy-cml /home/jovyan/cml_preprocessed_v3.nc \
  --strict-cml /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_STRICT_MINIMAL.nc \
  --radar-nc /home/jovyan/RADAR_S2021E2021_5MIN_CZ.nc \
  --estimator-script /home/jovyan/cml_estimate_and_grid_cml_radar_gauge_merged_parallel_timewindow_v2.py \
  --out-dir /home/jovyan/filter_strategy_rainfall_experiment \
  --tstart "2021-01-01 00:00:00" \
  --tend "2021-12-31 23:59:00" \
  --rain-variable R_mean_5min \
  --radar-var R_radar \
  --legacy-label "Legacy filter" \
  --strict-label "Strict link/sublink/sample QC" \
  --radar-wet-thr 0.1 \
  --min-rain 0 \
  --max-rain 100 \
  --link-chunk 32 \
  --time-chunk 2016 \
  --max-plot-points 500000 \
  --gridsize 50 \
  --overwrite-estimation

FILTER-STRATEGY RAINFALL EXPERIMENT
Legacy filtered input : /home/jovyan/cml_preprocessed_v3.nc
Legacy total loss     : trsl
Strict filtered input : /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_STRICT_MINIMAL.nc
Strict total loss     : trsl_qc
Radar                 : /home/jovyan/RADAR_S2021E2021_5MIN_CZ.nc
Estimator             : /home/jovyan/cml_estimate_and_grid_cml_radar_gauge_merged_parallel_timewindow_v2.py
Output directory      : /home/jovyan/filter_strategy_rainfall_experiment

IDENTICAL RAINFALL CONFIGURATION
  baseline             : pycml_constant
  k-R                  : ITU_2003
  WAA                  : pastorek_2021
  Pastorek A_max       : 14.0
  Pastorek zeta        : 0.55
  Pastorek d           : 0.1
  baseline pad         : 48.0 h
  min link length      : 0.5 km
  min valid samples    : 100
  estimation wet thr   : 0.00833 mm h-1
  final hexbin wet thr : radar > 0.1 mm h-1
  estimation window    : 2021-01-01 00:00:00 -> 2021-12-31 23:59:00

RUNNING
/opt/conda/bin/python -u /

In [ ]:
!python cml_estimate_and_grid_cml_radar_gauge_merged_parallel_timewindow.py \
  --cml-nc cml_preprocessed_v3.nc \
  --radar-nc RADAR_S2021E2021_5MIN_CZ.nc \
  --gauge-nc CHMI_10MIN_v1.nc \
  --estimation-nc cml_rainrate_baseline_kr_waa_methods_june.nc \
  --gridded-nc cml_rainfall_gridded_june.nc \
  --tstart "2020-01-01 00:00:00" \
  --tend "2021-12-31 23:59:00" \
  --grid-baseline-method pycml_constant \
  --grid-kr-model ITU_2003 \
  --grid-waa-method pastorek_2021 \
  --grid-time-chunk 288 \
  --gauge-time-chunk 144 \
  --batch 32 \
  --cell-chunk 20000 \
  --grid-workers 2 \
  --idw-query-workers 1 \
  --overwrite-estimation \
  --overwrite-grid

In [ ]:
!cd /home/jovyan

!OMP_NUM_THREADS=1 \
!MKL_NUM_THREADS=1 \
!OPENBLAS_NUM_THREADS=1 \
!NUMBA_NUM_THREADS=16 \
!python -u /home/jovyan/cml_dual_wetdry_pastorek_optimized.py \
  --input-nc /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_GRAF_RSTD_WETDRY.nc \
  --radar-nc /home/jovyan/RADAR_S2021E2021_5MIN_CZ.nc \
  --output-graf-nc /home/jovyan/CML_RAIN_PASTOREK_GRAF_1MIN.nc \
  --output-radar-nc /home/jovyan/CML_RAIN_PASTOREK_RADAR_5MIN.nc \
  --log-csv /home/jovyan/CML_RAIN_PASTOREK_PROCESSING_LOG.csv \
  --summary-json /home/jovyan/CML_RAIN_PASTOREK_RUN_SUMMARY.json \
  --time-dim time \
  --cml-dim cml_id \
  --sublink-dim sublink_id \
  --radar-time-dim time \
  --radar-cml-dim cml_id \
  --tsl-var tx \
  --rsl-var rx \
  --graf-wet-var graf_rstd_wet_1min \
  --graf-valid-var graf_rstd_valid_1min \
  --frequency-var frequency \
  --length-var length \
  --polarization-var polarization \
  --radar-var rr \
  --frequency-unit MHz \
  --length-unit km \
  --kr-model ITU_2005 \
  --tstart "2020-01-01 00:00:00" \
  --tend "2021-12-31 23:59:00" \
  --baseline-pad-hours 48 \
  --radar-scale 12 \
  --radar-wet-threshold 0.1 \
  --radar-timestamp-label start \
  --min-cml-valid-fraction 1.0 \
  --n-average-last-dry 1 \
  --pastorek-a-max 14 \
  --pastorek-zeta 0.55 \
  --pastorek-d 0.1 \
  --n-workers 16 \
  --link-batch 32 \
  --availability-scan-batch 256 \
  --compression-level 1 \
  --time-chunk-1min 1440 \
  --time-chunk-radar 288 \
  --output-link-chunk 8 \
  --sync-every-batches 4 \
  --overwrite

In [5]:
!cd /home/jovyan


!python -u /home/jovyan/cml_graf_blettner_publishable_pipeline_v2.py \
  --cml-qc-nc /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_GRAF_RSTD_WETDRY.nc \
  --radar-nc /home/jovyan/RADAR_S2021E2021_5MIN_CZ.nc \
  --gauge-nc /home/jovyan/CHMI_10MIN_v1.nc \
  --output-v2-nc /home/jovyan/CML_RAIN_PASTOREK_GRAF_1MIN_V2.nc \
  --out-dir /home/jovyan/GRAF_PASTOREK_BLETTNER_PUBLISHABLE_RESULTS \
  --time-dim time \
  --cml-dim cml_id \
  --sublink-dim sublink_id \
  --trsl-var trsl_qc \
  --graf-wet-var graf_rstd_wet_1min \
  --graf-valid-var graf_rstd_valid_1min \
  --link-pass-var qc_final_link_pass \
  --sublink-pass-var qc_final_sublink_pass \
  --qc-valid-var qc_final_valid \
  --frequency-var frequency \
  --length-var length \
  --polarization-var polarization \
  --site-a-lon site_a_longitude \
  --site-a-lat site_a_latitude \
  --site-b-lon site_b_longitude \
  --site-b-lat site_b_latitude \
  --frequency-unit MHz \
  --length-unit km \
  --kr-model ITU_2005 \
  --n-average-last-dry 1 \
  --pastorek-a-max 14 \
  --pastorek-zeta 0.55 \
  --pastorek-d 0.1 \
  --baseline-pad-hours 48 \
  --tstart "2020-01-01 00:00:00" \
  --tend "2021-12-31 23:59:00" \
  --n-workers 16 \
  --link-batch 8 \
  --compression-level 1 \
  --time-chunk-1min 10080 \
  --output-link-chunk 8 \
  --sync-every-batches 4 \
  --cml-rain-var R_cml \
  --cml-wet-var graf_wet \
  --cml-valid-var graf_wet_valid \
  --eval-link-batch 8 \
  --wet-aggregation any \
  --min-cml-valid-fraction 1.0 \
  --wet-threshold 0.1 \
  --hexbin-scope reference_wet \
  --radar-var rr \
  --radar-scale 12 \
  --radar-minutes 5 \
  --radar-timestamp-label start \
  --gauge-var rr \
  --gauge-scale 6 \
  --gauge-minutes 10 \
  --gauge-timestamp-label start \
  --projection EPSG:25833 \
  --max-gauge-distance-m 1000 \
  --spatial-batch 128 \
  --plot-max 150 \
  --gridsize 55 \
  --dpi 300 \
  --save-pdf \
  --overwrite

Running cml_graf_blettner_publishable_pipeline_v2.py version 2.0
GRAF V2 rainfall: 100%|█| 258/258 [14:35<00:00,  3.39s/batch, links 2056:2064 | 
reference evaluation: 100%|████████████████| 258/258 [11:34<00:00,  2.69s/batch]

=== BLETTNER / V2 FILTERING ===
Input CML links                         : 2,580
Links passing qc_final_link_pass        : 2,064
Links dropped by qc_final_link_pass     : 516
Static sublinks passing                 : 3,907
Static sublinks dropped                 : 221
Native observations retained            : 674,453,919
Native observations masked              : 3,670,844,001

=== RADAR POPULATION ===
V2 links with radar ID mapping          : 2,064
Links used in radar contingency         : 1,326
Links used in radar hexbin              : 1,207
Radar contingency pooled N              : 86,636,751
Radar hexbin pooled N                   : 4,861,776

=== GAUGE POPULATION ===
Gauge stations in source                : 430
Unique gauges used within distance      : 131
V

In [ ]:
!cd /home/jovyan

!mkdir -p /home/jovyan/CML_RAIN_MULTI_WAA_ITU_2020_2021

!OMP_NUM_THREADS=1 \
!MKL_NUM_THREADS=1 \
!OPENBLAS_NUM_THREADS=1 \
!NUMBA_NUM_THREADS=16 \
!python -u /home/jovyan/cml_multi_waa_itu_fullperiod_optimized.py \
  --input-nc /home/jovyan/CML_S2020E2021_1MIN_CZ_QC_GRAF_RSTD_WETDRY.nc \
  --radar-nc /home/jovyan/RADAR_S2021E2021_5MIN_CZ.nc \
  --output-dir /home/jovyan/CML_RAIN_MULTI_WAA_ITU_2020_2021 \
  --filename-prefix CML_RAIN \
  --time-dim time \
  --cml-dim cml_id \
  --sublink-dim sublink_id \
  --radar-time-dim time \
  --radar-cml-dim cml_id \
  --tsl-var tx \
  --rsl-var rx \
  --graf-wet-var graf_rstd_wet_1min \
  --graf-valid-var graf_rstd_valid_1min \
  --frequency-var frequency \
  --length-var length \
  --polarization-var polarization \
  --radar-var rr \
  --frequency-unit MHz \
  --length-unit km \
  --graf-tstart "2020-01-01 00:00:00" \
  --graf-tend "2021-12-31 23:59:00" \
  --baseline-pad-hours 48 \
  --radar-scale 12 \
  --radar-wet-threshold 0.1 \
  --radar-timestamp-label start \
  --min-cml-valid-fraction 1.0 \
  --n-average-last-dry 1 \
  --kr-models ITU_2003 ITU_2005 \
  --waa-methods none leijnse_2008 pastorek_2021 schleiss_2013 \
  --leijnse-gamma 1.4714285714285715e-05 \
  --leijnse-l-antenna 0.0043 \
  --leijnse-delta 0.36 \
  --pastorek-a-max 14 \
  --pastorek-zeta 0.55 \
  --pastorek-d 0.1 \
  --schleiss-waa-max 2 \
  --schleiss-delta-t 1 \
  --schleiss-tau 15 \
  --n-workers 16 \
  --link-batch 8 \
  --availability-scan-batch 256 \
  --compression-level 1 \
  --time-chunk-1min 10080 \
  --time-chunk-radar 2016 \
  --output-link-chunk 8 \
  --overwritey-batches 4 \